# Cell-Type Annotation via Logistic Regression (LUCA Reference)

Trains a logistic regression classifier on the published LUCA lung cancer
reference atlas (Salcher et al. 2022) restricted to the HVG overlap with our
data, predicts cell types on our scVI-clustered test data, and explores
EGFR-mutation distribution across predicted cell types.

## Setup

In [ ]:
import os
import pickle
import time

os.environ["NUMBA_CACHE_DIR"] = "/path/to/tmp"
os.environ["MPLCONFIGDIR"] = "/path/to/tmp"

import numpy as np
import scanpy as sc
from sklearn.linear_model import LogisticRegression

sc.set_figure_params(figsize=(8, 8))

## Config

In [ ]:
luca_ref_path = "/path/to/public/full_atlas_hvg_integrated_scvi_integrated_scanvi.h5ad"
test_input_path = "/path/to/output/S4_scVI_integrated_adata_cluster_latent10.h5ad"

model_output_path = "/path/to/results/log_regression/logreg_LUCAref_train_HVGs.pkl"
predictions_output_csv = "/path/to/results/log_regression/logreg_LUCAref_train_HVGs_predictions.csv"
test_output_path = "/path/to/output/S5a_scVI_integrated_adata_cluster_latent10_logreg.h5ad"

n_top_hvgs = 6000
logreg_params = dict(max_iter=5000, n_jobs=-1, random_state=0, C=0.2)


## Load reference atlas and test data

In [ ]:
adata_LUCAref = sc.read(luca_ref_path)
adata_LUCAref

In [ ]:
adata_test = sc.read(test_input_path)
adata_test.X = adata_test.layers["raw_counts"].copy()
adata_test

## Identify HVGs in test data and restrict to the train/test gene overlap

The reference atlas already ships with its own HVG set; we identify HVGs on our test data and later intersect the two gene sets.

In [ ]:
sc.pp.highly_variable_genes(adata_test, n_top_genes=n_top_hvgs, flavor="seurat_v3")
sc.pl.highly_variable_genes(adata_test)

adata_test = adata_test[:, adata_test.var["highly_variable"]].copy()
print("Total number of cells: {:d}".format(adata_test.n_obs))
print("Total number of genes: {:d}".format(adata_test.n_vars))

In [ ]:
adata_train = adata_LUCAref.copy()
del adata_LUCAref  # free memory; the full reference isn't needed again

# normalize, log-transform, and scale both train and test sets
sc.pp.normalize_total(adata_train, target_sum=1e4)
sc.pp.log1p(adata_train)
sc.pp.scale(adata_train, max_value=10)

sc.pp.normalize_total(adata_test, target_sum=1e4)
sc.pp.log1p(adata_test)
sc.pp.scale(adata_test, max_value=10)

In [ ]:
# sorting is important for consistent gene ordering between train and test
common_genes = sorted(set(adata_train.var_names) & set(adata_test.var_names))
print(
    f"{len(common_genes)} genes are shared between the reference and test data\n"
    f"{adata_train.n_vars} genes in reference data\n"
    f"{adata_test.n_vars} genes in test data"
)

adata_train_common = adata_train[:, common_genes].copy()
adata_test_common = adata_test[:, common_genes].copy()
assert list(adata_test_common.var_names) == list(adata_train_common.var_names)

## Train logistic regression classifier

In [ ]:
celltypes_train = adata_train_common.obs["cell_type"]

start = time.time()
logreg = LogisticRegression(**logreg_params)
logreg.fit(adata_train_common.X, celltypes_train)
print(f"Training took {time.time() - start:.1f}s")

with open(model_output_path, "wb") as f:
    pickle.dump(logreg, f)

## Predict cell types on test data

In [ ]:
predictions = logreg.predict(adata_test_common.X)
probabilities = logreg.predict_proba(adata_test_common.X)

adata_test_common.obs["celltype_predictions"] = predictions
adata_test_common.obs["probabilities"] = [
    probabilities[i, logreg.classes_ == pred][0] for i, pred in enumerate(predictions)
]

## Transfer predictions back and save

In [ ]:
adata_test.obs["celltype_predictions"] = adata_test_common.obs.loc[adata_test.obs_names, "celltype_predictions"]
adata_test.obs["probabilities"] = adata_test_common.obs.loc[adata_test.obs_names, "probabilities"]

adata_test.obs[["celltype_predictions", "probabilities"]].to_csv(predictions_output_csv)
adata_test.write(test_output_path)
adata_test

## Visualize predictions

In [ ]:
sc.pl.umap(
    adata_test, color=["leiden_scvi", "celltype_predictions", "probabilities"],
    legend_fontsize="xx-small", ncols=2, size=4,
    save="_scVI_latent10_clusters_LogReg.pdf",
)